# 01 · ETL : consolidation des ventes et des autres tables de faits

**Projet 4 : Analyse des données de vente et retours** · Sprint 1 · US09

**Objectif** : transformer les fichiers de ventes bruts en **une seule table typée**, prête à être jointe aux dimensions, puis charger les trois autres tables de faits extraites de Contoso (stock, objectifs, ventes en ligne détaillées).

**Période** : les trois années étudiées, 2019 à 2021. Le périmètre de l'énoncé est 2020 et 2021 ; 2019 est l'année de référence, chargée avec les deux autres pour pouvoir dire d'où vient la situation.

| Étape | Ce qu'on fait |
|---|---|
| 1. Chargement et typage | Lire les fichiers trimestriels, convertir les dates et les montants, les empiler |
| 2. Réconciliation | Vérifier qu'aucune ligne n'a été perdue ni ajoutée, et que tout a été converti |
| 3. Renommage | Donner aux colonnes des noms simples, sans accent ni espace |
| 4. Calendrier | Ajouter l'année, le trimestre et le mois |
| 5. Canal et devise | Rattacher à chaque vente son canal de vente (IdVente = SalesKey) |
| 6. Contrôles finaux | Clé unique, aucune valeur manquante, types corrects |
| 7. Enregistrement | Écrire `data/interim/ventes_consolidees.parquet` |
| 8. Autres tables de faits | Charger le stock, les objectifs et les ventes en ligne détaillées, vérifier qu'aucune ligne n'est perdue, écrire `data/interim` |

**Lit dans** : `data/raw/contoso/Sales`, `data/raw/contoso/Plan`, `data/raw/contoso/Stock`
**Écrit dans** : `data/interim` (`ventes_consolidees`, `stock`, `objectifs`, `ventes_en_ligne`)

Le plan stratégique (charges) n'est pas chargé ici : il a son propre notebook, `05b_charges`, qui le lit, le met en forme et le contrôle.

**Fonctions partagées** : toutes les transformations viennent de `src/etl.py`. Ce notebook les enchaîne, affiche les contrôles et explique les choix.

À ce stade, les données sont **transformées mais pas nettoyées** : les anomalies éventuelles (incohérences de montants, valeurs aberrantes) sont encore là. Elles seront diagnostiquées dans `03_diagnostic` et traitées dans `04_nettoyage`.

## 0. Préparation

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
# On remonte de dossier en dossier jusqu'à trouver celui qui contient src/ et data/.
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent

# Ajouter la racine aux dossiers où Python cherche les modules
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config -> chemins des dossiers et paramètres
#   etl    -> fonctions de lecture, de conversion et de consolidation
from src import config, etl

config.creer_dossiers()

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Racine du projet :", config.ROOT)
print("Lecture  :", config.SALES)
print("Écriture :", config.INTERIM)

## 1. Chargement et typage

In [ ]:
# ---------------------------------------------------------------------------
# Lire, convertir et empiler tous les fichiers de ventes
# ---------------------------------------------------------------------------
# etl.charger_ventes() fait, pour chaque fichier de data/raw/contoso/Sales :
#   1. etl.lire_csv      : lecture en texte brut, avec détection de l'encodage
#   2. etl.typer_ventes  : dates -> datetime, montants "$1,062.60" -> 1062.60,
#                          quantités et identifiants -> nombres
#   3. ajout d'une colonne "fichier" (origine de chaque ligne, pour la traçabilité)
# puis empile le tout en une seule table.
#
# Elle renvoie :
#   ventes   -> toutes les ventes, typées
#   controle -> une ligne par fichier : encodage, nombre de lignes, format de date,
#               nombre de valeurs non converties
ventes, controle = etl.charger_ventes()

display(controle)
print(f"Table consolidée : {len(ventes):,} lignes, {ventes.shape[1]} colonnes".replace(",", " "))

## 2. Réconciliation avec les fichiers bruts

Une consolidation ne doit **rien perdre et rien inventer**. On compare donc la table consolidée aux fichiers d'origine, relus indépendamment.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Compter les lignes de chaque fichier brut, sans passer par etl
# ---------------------------------------------------------------------------
# On relit chaque fichier comme un simple texte et on compte ses lignes,
# moins 1 pour l'en-tête. C'est un comptage indépendant de pandas :
# s'il donne le même nombre, la lecture n'a perdu aucune ligne.
reconciliation = []
for f in etl.lister_fichiers_ventes():
    # "rb" : lecture en binaire, donc sans se soucier de l'encodage.
    # On compte les lignes non vides.
    with open(f, "rb") as fichier:
        lignes_brutes = sum(1 for ligne in fichier if ligne.strip()) - 1

    # Nombre de lignes de ce fichier dans la table consolidée
    lignes_consolidees = int((ventes.fichier == f.name).sum())

    reconciliation.append({
        "fichier": f.name,
        "lignes dans le fichier": lignes_brutes,
        "lignes consolidées": lignes_consolidees,
        "écart": lignes_consolidees - lignes_brutes,
    })

reconciliation = pd.DataFrame(reconciliation)
display(reconciliation)


# ---------------------------------------------------------------------------
# 2. Arrêter le notebook si quelque chose ne colle pas
# ---------------------------------------------------------------------------
# assert vérifie une condition : si elle est fausse, le notebook s'arrête avec le message.
# On préfère une erreur visible maintenant à des données fausses enregistrées en silence.
assert (reconciliation["écart"] == 0).all(), "Des lignes ont été perdues ou ajoutées à la consolidation"
assert controle.valeurs_non_converties.sum() == 0, "Certaines valeurs n'ont pas pu être converties : voir le tableau de contrôle"
print("Réconciliation réussie : aucune ligne perdue, aucune valeur non convertie")

## 3. Renommage des colonnes

Les noms d'origine contiennent des accents et des espaces (« Quantité de vente »). On les remplace par des noms simples en minuscules (`quantite_vente`), plus faciles à écrire dans le code et sans risque dans Power BI, SQL ou Parquet.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Table de correspondance entre anciens et nouveaux noms
# ---------------------------------------------------------------------------
# etl.RENOMMAGE_VENTES est un dictionnaire  ancien nom -> nouveau nom.
# On l'affiche sous forme de tableau pour le reporter dans le dictionnaire de données.
renommage = pd.DataFrame(
    etl.RENOMMAGE_VENTES.items(), columns=["nom d'origine", "nom après ETL"])
display(renommage)

# ---------------------------------------------------------------------------
# 2. Appliquer le renommage
# ---------------------------------------------------------------------------
# Les colonnes absentes du dictionnaire (ici "fichier") gardent leur nom.
ventes = etl.renommer_colonnes(ventes)
print("Colonnes :", list(ventes.columns))

## 4. Colonnes de calendrier

In [ ]:
# ---------------------------------------------------------------------------
# Ajouter l'année, le trimestre et le mois, déduits de la date
# ---------------------------------------------------------------------------
# Ces trois colonnes servent aux regroupements simples (ventes par trimestre...).
# Le calendrier complet (jour de la semaine, jours fériés, périodes COVID-19)
# sera construit dans 02_flocon, sous forme d'une vraie dimension date.
ventes = etl.ajouter_calendrier(ventes)

# Contrôle : nombre de ventes par année et par trimestre.
# unstack() place les trimestres en colonnes ; fill_value=0 met 0 pour les trimestres absents.
display(ventes.groupby(["annee", "trimestre"]).size().unstack(fill_value=0))

## 5. Canal de vente et devise

Nos fichiers n'ont ni canal ni devise. Comme **IdVente = FactSales.SalesKey**, on récupère ces deux informations dans le fichier extrait par `src/extraction.py`.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Jointure avec le fichier des clés
# ---------------------------------------------------------------------------
# etl.joindre_canal_devise fait une jointure "à gauche" : toutes nos ventes sont
# conservées, et on leur ajoute id_canal et id_devise quand la clé est trouvée.
# Elle vérifie aussi qu'une clé n'apparaît qu'une fois dans le fichier des clés
# (sinon la jointure dupliquerait des ventes).
lignes_avant = len(ventes)
ventes, sans_correspondance = etl.joindre_canal_devise(ventes)

# ---------------------------------------------------------------------------
# 2. Contrôles de la jointure
# ---------------------------------------------------------------------------
assert len(ventes) == lignes_avant, "La jointure a changé le nombre de lignes"
print(f"Ventes sans canal : {sans_correspondance}")

# Répartition des ventes par canal et par devise.
# normalize=True donne des proportions ; on les multiplie par 100 pour avoir des %.
print("\nRépartition par canal (%) :")
print((ventes.id_canal.value_counts(normalize=True) * 100).round(1).to_string())
print("\nDevises présentes :", sorted(int(d) for d in ventes.id_devise.dropna().unique()))

# ---------------------------------------------------------------------------
# 3. Retirer la devise si elle ne contient qu'une seule valeur
# ---------------------------------------------------------------------------
# Une colonne qui a la même valeur partout n'apporte aucune information :
# on la retire pour alléger la table, et on le signale.
if ventes.id_devise.nunique() <= 1:
    ventes = ventes.drop(columns="id_devise")
    print("Une seule devise : colonne id_devise retirée (DimCurrency écartée)")

# Les clés jointes sont des décimaux (à cause des éventuelles valeurs manquantes) :
# "Int64" est le type entier de pandas qui accepte les valeurs manquantes.
ventes["id_canal"] = ventes["id_canal"].astype("Int64")

## 6. Contrôles finaux

In [ ]:
# ---------------------------------------------------------------------------
# 1. Ordonner les colonnes : clés, calendrier, mesures, puis traçabilité
# ---------------------------------------------------------------------------
ordre = (
    ["id_vente", "date", "annee", "trimestre", "mois",
     "id_boutique", "id_produit", "id_promotion", "id_canal"]
    + ["quantite_vente", "prix_unitaire", "cout_unitaire",
       "montant_ventes", "cout_total",
       "quantite_remise", "montant_remise",
       "quantite_retour", "montant_retour"]
    + ["fichier"]
)
# On garde aussi toute colonne qui ne serait pas dans la liste (sécurité)
ventes = ventes[[c for c in ordre if c in ventes.columns]
                + [c for c in ventes.columns if c not in ordre]]

# Trier par date puis par identifiant, pour un fichier toujours identique d'une exécution à l'autre
ventes = ventes.sort_values(["date", "id_vente"]).reset_index(drop=True)


# ---------------------------------------------------------------------------
# 2. Tableau de contrôle : type, valeurs manquantes et valeurs distinctes par colonne
# ---------------------------------------------------------------------------
controle_final = pd.DataFrame({
    "type": ventes.dtypes.astype(str),
    "valeurs manquantes": ventes.isna().sum(),
    "valeurs distinctes": ventes.nunique(),
})
display(controle_final)


# ---------------------------------------------------------------------------
# 3. Règles que la table doit respecter avant d'être enregistrée
# ---------------------------------------------------------------------------
assert ventes.id_vente.is_unique, "Des id_vente sont en double"
assert ventes.date.notna().all(), "Des dates sont manquantes"
assert str(ventes.date.dtype).startswith("datetime"), "La colonne date n'est pas au format date"
print("Contrôles réussis : clé unique, dates complètes et au bon format")

## 7. Enregistrement dans data/interim

In [ ]:
# ---------------------------------------------------------------------------
# 1. Enregistrer la table consolidée
# ---------------------------------------------------------------------------
# etl.enregistrer_interim écrit data/interim/<nom>.parquet et renvoie le chemin.
# Le format Parquet conserve les types (dates, entiers, décimaux), contrairement
# au CSV où tout redevient du texte. Il est aussi plus compact et plus rapide à relire.
chemin = etl.enregistrer_interim(ventes, "ventes_consolidees")

# ---------------------------------------------------------------------------
# 2. Relire le fichier pour vérifier qu'il est identique à la table en mémoire
# ---------------------------------------------------------------------------
relu = pd.read_parquet(chemin)
assert len(relu) == len(ventes), "Le fichier relu n'a pas le même nombre de lignes"
assert list(relu.columns) == list(ventes.columns), "Le fichier relu n'a pas les mêmes colonnes"

# ---------------------------------------------------------------------------
# 3. Résumé
# ---------------------------------------------------------------------------
print("Fichier enregistré :", chemin)
print(f"Taille : {chemin.stat().st_size / 1e6:.1f} Mo")
print(f"Lignes : {len(relu):,}".replace(",", " "))
print(f"Période : {relu.date.min().date()} au {relu.date.max().date()}")
print(f"Chiffre d'affaires total : {relu.montant_ventes.sum():,.0f} $".replace(",", " "))
relu.head()

## 8. Les autres tables de faits

Trois tables viennent compléter les ventes. `src/extraction.py` les a écrites au format Parquet : leurs types sont déjà corrects (dates en dates, quantités en entiers, montants en décimaux), il n'y a donc **aucune conversion à faire**. Le travail se limite à trois choses :

| Opération | Pourquoi |
|---|---|
| Renommer les colonnes | Mêmes noms que dans les ventes (`id_boutique`, `id_produit`, `date`), pour que les jointures soient immédiates |
| Vérifier le nombre de lignes | Comparer la table chargée au fichier brut, relu indépendamment |
| Vérifier la clé | Une ligne par boutique, produit et semaine (stock) ; par boutique, produit, mois et scénario (objectifs) ; par ligne de commande (ventes en ligne) |

Comme pour les ventes, ces tables sont **transformées mais pas nettoyées** : leur diagnostic est fait dans `05c_stock_objectifs`.

In [ ]:
import pyarrow.parquet as pq

# ---------------------------------------------------------------------------
# 1. Charger les trois tables (fonctions de src/etl.py)
# ---------------------------------------------------------------------------
stock = etl.charger_stock()
objectifs = etl.charger_objectifs()
ventes_en_ligne, controle_en_ligne = etl.charger_ventes_en_ligne()

AUTRES = {
    # nom du fichier interim : (table, fichiers bruts, clé qui doit être unique)
    "stock": (stock, [config.STOCK / config.FICHIER_STOCK], ["date", "id_boutique", "id_produit"]),
    "objectifs": (objectifs, [config.PLAN / config.FICHIER_OBJECTIFS], ["date", "id_boutique", "id_produit", "id_scenario"]),
    "ventes_en_ligne": (ventes_en_ligne, sorted(config.SALES_CONTOSO.glob(config.MOTIF_FICHIERS_EN_LIGNE)), ["id_vente_en_ligne"]),
}

# ---------------------------------------------------------------------------
# 2. Réconciliation : la table chargée a-t-elle autant de lignes que les fichiers bruts ?
# ---------------------------------------------------------------------------
# pq.ParquetFile(...).metadata.num_rows lit le nombre de lignes inscrit dans le fichier,
# sans passer par pandas : c'est un comptage indépendant de notre chargement.
bilan = []
for nom, (table, fichiers_bruts, cle) in AUTRES.items():
    lignes_brutes = sum(pq.ParquetFile(f).metadata.num_rows for f in fichiers_bruts)
    bilan.append({
        "table": nom,
        "fichiers bruts": len(fichiers_bruts),
        "lignes dans les fichiers": lignes_brutes,
        "lignes chargées": len(table),
        "écart": len(table) - lignes_brutes,
        "colonnes": table.shape[1],
        "valeurs manquantes": int(table.isna().sum().sum()),
        "doublons sur la clé": int(table.duplicated(cle).sum()),
        "du": table.date.min().date(),
        "au": table.date.max().date(),
    })
bilan = pd.DataFrame(bilan).set_index("table")
display(bilan)

assert (bilan["écart"] == 0).all(), "Des lignes ont été perdues ou ajoutées au chargement"
assert (bilan["doublons sur la clé"] == 0).all(), "Une clé est répétée : voir le tableau"
assert (bilan["valeurs manquantes"] == 0).all(), "Des valeurs manquantes sont apparues au chargement"
print("Réconciliation réussie : aucune ligne perdue, aucune clé répétée, aucune valeur manquante")

# ---------------------------------------------------------------------------
# 3. Les clés de ces tables existent-elles dans nos ventes ?
# ---------------------------------------------------------------------------
# Une boutique absente de nos ventes signalerait une erreur d'extraction.
# Un produit absent de nos ventes n'est pas une erreur : il peut être en stock sans s'être vendu.
boutiques_ventes, produits_ventes = set(ventes.id_boutique), set(ventes.id_produit)
cles = pd.DataFrame({nom: {
    "boutiques": table.id_boutique.nunique(),
    "boutiques absentes des ventes": len(set(table.id_boutique) - boutiques_ventes),
    "produits": table.id_produit.nunique(),
    "produits jamais vendus": len(set(table.id_produit) - produits_ventes),
} for nom, (table, _, _) in AUTRES.items()}).T
display(cles)
assert (cles["boutiques absentes des ventes"] == 0).all(), "Une table contient une boutique hors de notre périmètre"

# ---------------------------------------------------------------------------
# 4. Enregistrer dans data/interim
# ---------------------------------------------------------------------------
for nom, (table, _, cle) in AUTRES.items():
    # Tri sur la clé : le fichier est identique d'une exécution à l'autre
    chemin_table = etl.enregistrer_interim(table.sort_values(cle).reset_index(drop=True), nom)
    assert pq.ParquetFile(chemin_table).metadata.num_rows == len(table), f"{nom} : le fichier relu n'a pas le même nombre de lignes"
    print(f"Fichier enregistré : {chemin_table.name:28s} {len(table):>9,} lignes · {chemin_table.stat().st_size / 1e6:5.1f} Mo".replace(",", " "))

## Suite

`02_flocon` lira `data/interim/ventes_consolidees.parquet`, réalignera les dates des dimensions de +12 ans, construira le calendrier 2019-2021 et joindra les dimensions aux ventes.

`05c_stock_objectifs` lira `stock`, `objectifs` et `ventes_en_ligne` pour en faire le diagnostic et calculer les variables d'assortiment et d'écart aux objectifs.